<!-- Cache bust 17_typing_and_metaprogramming_notebook -->

# Typing & Metaprogramming

***

### 🔹 1. Advanced Type Hinting
Python is a dynamically typed language, but type hinting (introduced in Python 3.5) allows developers to specify variable and function signatures. This makes code clean, self-documenting, and searchable by static analyzers like `mypy` or IDE auto-completers.

#### Core components from the `typing` module:
* **`Union[A, B]`**: The variable/parameter can be of type A or B (or `A | B` in Python 3.10+).
* **`Optional[T]`**: The variable can be of type T or `None` (equivalent to `Union[T, None]`).
* **`Callable[[Arg1Type, Arg2Type], ReturnType]`**: Hints that the parameter is a function.
* **`Any`**: Disables type checking for that variable.
* **`Protocol`**: Used for structural subtyping (static duck typing).

In [ ]:
from typing import List, Dict, Union, Optional, Callable

# Simple type hints
def process_scores(scores: List[float], weight: Optional[float] = None) -> List[float]:
    factor = weight if weight is not None else 1.0
    return [score * factor for score in scores]

# Hinting a function parameter (Callable)
def execute_operation(func: Callable[[int, int], int], a: int, b: int) -> int:
    return func(a, b)

sum_func: Callable[[int, int], int] = lambda x, y: x + y
print("Operation Result:", execute_operation(sum_func, 5, 12))

***

### 🔹 2. Object Serialization (JSON, Pickle, Dill)
**Serialization** (pickling) is the process of converting a Python object structure into a byte stream (to save to disk or send over a network). **Deserialization** (unpickling) is the inverse operation.

| Tool | Format | Handles Custom Objects / Classes? | Security |
| :--- | :---: | :---: | :--- |
| **`json`** | Text (JSON) | No (Basic types only) | Safe |
| **`pickle`** | Binary | Yes (Standard objects) | **⚠️ Dangerous** (Executing arbitrary code on load) |
| **`dill`** | Binary | Yes (Lambda functions, closures, dynamic functions) | **⚠️ Dangerous** |

In [ ]:
import pickle

class ModelConfig:
    def __init__(self, name, parameters):
        self.name = name
        self.parameters = parameters

config = ModelConfig("XGBoost", {"depth": 6, "lr": 0.1})

# Serialize object to bytes
serialized_bytes = pickle.dumps(config)
print("Serialized bytes snippet:", serialized_bytes[:40])

# Deserialize object back
loaded_config = pickle.loads(serialized_bytes)
print("Deserialized Model Name:", loaded_config.name)
print("Deserialized Hyperparameters:", loaded_config.parameters)

***

### 🔹 3. Metaprogramming: type as a Class Factory
In Python, **everything is an object, including classes**. 
* The class of a class is **`type`**.
* Calling `type(object)` returns the type of the object.
* You can also use `type()` to create new classes dynamically at runtime!
  - **Syntax:** `type(classname, tuple_of_parent_classes, dictionary_of_attributes)`

In [ ]:
# Dynamic Class Creation
def greet(self):
    return f"Hello, I am {self.name}."

# Create class 'Robot' dynamically
Robot = type("Robot", (object,), {"name": "Optimus Prime", "greet": greet})

# Instantiate
bot = Robot()
print("Class created dynamically:", Robot)
print("Instance Name:", bot.name)
print("Greet behavior:", bot.greet())

***

### 🔹 4. Instantiation Control: __new__ vs __init__
When you instantiate a class (e.g. `obj = MyClass()`), Python performs two steps:
1. **`__new__(cls, *args, **kwargs)`**: A static method that actually allocates memory and creates the empty object instance.
2. **`__init__(self, *args, **kwargs)`**: An instance method that initializes the attributes of that created instance.

#### 🧠 Real-world usage: The Singleton Pattern
A Singleton pattern ensures that a class has **only one instance** globally. We intercept creation in `__new__` to implement this.

In [ ]:
class DatabaseConnection:
    _instance = None # Class-level cache for the singleton instance
    
    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            print("[DatabaseConnection] Creating a new database connection instance...")
            # Allocate memory using parent object's __new__
            cls._instance = super().__new__(cls)
        return cls._instance
        
    def __init__(self, db_name):
        # Note: __init__ runs every time class is instantiated
        self.db_name = db_name

conn1 = DatabaseConnection("Prod_DB")
conn2 = DatabaseConnection("Stage_DB")

print("Is conn1 the same object as conn2?", conn1 is conn2)
print("conn1 DB name:", conn1.db_name)
print("conn2 DB name:", conn2.db_name) # Both show Stage_DB because __init__ re-ran on the same instance

***

### 🔹 5. Custom Metaclasses
* A **Metaclass** is the blueprint for a class (just as a class is the blueprint for an object).
* It allows us to control the creation, modifications, and rules of classes.
* Custom metaclasses inherit from **`type`**.

In [ ]:
# A metaclass that enforces all attribute names in subclasses to be lowercase
class StrictNamingMetaclass(type):
    def __new__(mcs, name, bases, attrs):
        for attr_name in attrs.keys():
            # Skip magic dunder methods
            if not attr_name.startswith("__") and not attr_name.islower():
                raise TypeError(f"Attribute/Method name '{attr_name}' must be completely lowercase!")
        return super().__new__(mcs, name, bases, attrs)

try:
    class MySmartClass(metaclass=StrictNamingMetaclass):
        def good_method(self):
            pass
            
        def BAD_METHOD(self): # Will raise TypeError on class load
            pass
except TypeError as e:
    print("Class definition failed as expected:", e)

***

### 🔹 6. The Descriptor Protocol
Descriptors allow us to customize attribute lookup, modification, and deletion behaviors dynamically.
* To create a descriptor, write a class that implements one or more of:
  - `__get__(self, instance, owner)`
  - `__set__(self, instance, value)`
  - `__delete__(self, instance)`
* Descriptors are the underlying technology behind Python's `@property`, `@classmethod`, and `@staticmethod`.

In [ ]:
class NonNegative:
    def __init__(self, name):
        self.name = name
        
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.name, 0)
        
    def __set__(self, instance, value):
        if value < 0:
            raise ValueError(f"Attribute '{self.name}' cannot be negative!")
        instance.__dict__[self.name] = value

class Product:
    # Set up class-level descriptors
    price = NonNegative("price")
    quantity = NonNegative("quantity")
    
    def __init__(self, name, price, quantity):
        self.name = name
        self.price = price       # Intercepted by __set__
        self.quantity = quantity # Intercepted by __set__

prod = Product("Laptop", 1200, 5)

try:
    prod.price = -100 # Raises ValueError
except ValueError as e:
    print("Descriptor blocked invalid assignment:", e)

***

## 📝 Practice Questions


### 🟢 Easy Level


In [ ]:
#Q1 Declare type hints for an integer and float variable.

In [ ]:
#Q2 Declare type hints for function parameters and return type.

In [ ]:
#Q3 Type hint a list containing only string elements using `List`.

In [ ]:
#Q4 Type hint a dictionary mapping strings to integers using `Dict`.

In [ ]:
#Q5 Type hint a Tuple containing fixed coordinate element types.

In [ ]:
#Q6 Use `Union` to type hint a variable that can be float or integer.

In [ ]:
#Q7 Use `Optional` to type hint a variable that can be string or None.

In [ ]:
#Q8 Inspect attributes of an object dynamically using `dir()`.

In [ ]:
#Q9 Set an attribute dynamically on a class object using `setattr()`.

In [ ]:
#Q10 Retrieve an attribute dynamically from a class object using `getattr()`.

### 🟡 Medium Level


In [ ]:
#Q11 Type hint Callable parameters (functions passing functions).

In [ ]:
#Q12 Implement a descriptor class validation getter and setter.

In [ ]:
#Q13 Create class dynamically using type() constructor with name 'SubClass'.

In [ ]:
#Q14 Create custom metaclass that prints class name when class is compiled.

In [ ]:
#Q15 Type hint Generic class mapping type variable using `TypeVar`.

In [ ]:
#Q16 Check if attribute exists on class object using `hasattr()`.

In [ ]:
#Q17 Implement Singleton pattern utilizing custom Metaclass.

In [ ]:
#Q18 Type hint class instance constructor parameters referencing its own class type.

In [ ]:
#Q19 Type hint returning Union of dict or list from calculation.

In [ ]:
#Q20 Check memory addresses of dynamic classes created using `type()` vs standard classes.

In [ ]:
#Q21 Type hint arguments checking they match custom interface or protocol.

In [ ]:
#Q22 Demonstrate dynamic dictionary type casting checking keys at runtime.

### 🔴 Hard Level


In [ ]:
#Q23 Implement a descriptor class validating that attribute values assigned are not empty strings.

In [ ]:
#Q24 Implement custom metaclass that automatically registers classes in global list registry.

In [ ]:
#Q25 Metaclass enforcing class attribute validations (e.g. classes must define field 'name').

In [ ]:
#Q26 Implement custom read-only attribute descriptor.

In [ ]:
#Q27 Implement dynamic attribute lookup intercepting missing attributes using `__getattr__`.

In [ ]:
#Q28 Type hint complex nested configurations using TypeAlias and Literal types.

In [ ]:
#Q29 Metaclass that intercepts subclass creations and dynamically adds timestamps to attributes.

In [ ]:
#Q30 Implement type casting validator function that checks types recursively for lists of dictionaries.